In [1]:
## fetching the dataset
import pandas as pd 
import os 

os.makedirs('data',exist_ok=True)
DATA_URL = 'https://raw.githubusercontent.com/ashutoshmakone/Bank-Marketing-Dataset-Machine-Learning/refs/heads/main/bank.csv'

bank_df = pd.read_csv(DATA_URL)

## saving the dataframe into the local dir
bank_df.to_csv('data/bank.csv',index=False)

## reading the head of the dataset
bank_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [2]:
## Shape of the dataframe
print(f'Rows: {bank_df.shape[0]}, Columns: {bank_df.shape[1]}')

Rows: 11162, Columns: 17


In [3]:
## Checking missing values and duplicates 
print(f'Missing Values: \n{bank_df.isna().sum()}')
print('-'*15)
print(f"Duplicates found: {bank_df.duplicated().sum()}")

Missing Values: 
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64
---------------
Duplicates found: 0


- Zero missing values and duplicates found. Dataset is clean.

In [4]:
## Target Distribution
bank_df['deposit'].value_counts(normalize=True) * 100

deposit
no     52.616019
yes    47.383981
Name: proportion, dtype: float64

In [5]:
import plotly.express as px

fig = px.histogram(
    data_frame=bank_df,
    x='deposit',
    color='deposit',
    title=f"Target distribution: Subscribe term deposit or not?",
    opacity=0.45
)

fig.update_layout(
    title_x = 0.5,
    xaxis_title = 'Deposit',
    yaxis_title = 'Number of Customers',
    width = 900,
    height = 600
)

fig.show()

- Target distribution is slightly imbalanced. However the degree of imbalance is not too high to implement SMOTE technique in later stages of analysis

In [6]:
## Checking Unique values column-wise
for column in bank_df.columns:
    print(f"{column} has unique values: {bank_df[column].unique()}")
    print('='*15)
    

age has unique values: [59 56 41 55 54 42 60 37 28 38 30 29 46 31 35 32 49 43 26 40 33 23 48 45
 36 52 53 39 57 51 44 24 50 27 34 47 25 58 61 68 75 22 69 66 85 72 90 67
 71 21 74 65 62 83 70 76 77 19 73 63 20 78 95 64 79 82 18 86 84 87 92 81
 80 93 88 89]
job has unique values: <ArrowStringArray>
[       'admin.',    'technician',      'services',    'management',
       'retired',   'blue-collar',    'unemployed',  'entrepreneur',
     'housemaid',       'unknown', 'self-employed',       'student']
Length: 12, dtype: str
marital has unique values: <ArrowStringArray>
['married', 'single', 'divorced']
Length: 3, dtype: str
education has unique values: <ArrowStringArray>
['secondary', 'tertiary', 'primary', 'unknown']
Length: 4, dtype: str
default has unique values: <ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str
balance has unique values: [2343   45 1270 ... 1594 2593 -134]
housing has unique values: <ArrowStringArray>
['yes', 'no']
Length: 2, dtype: str
loan has unique values: <

In [7]:
## Checking the frequency of unknown and -1 from categorical columns and pdays respectively.
import warnings
warnings.filterwarnings('ignore')

for column in bank_df.select_dtypes(include='object').columns:
    n_unknown = (bank_df[column] == 'unknown').sum()

    if n_unknown > 0:
        print(f"{column} has unique values: {n_unknown} || {n_unknown/len(bank_df) * 100:.2f}%")

print()
print(f"pdays=-1 (customer was not contacted in previous campaign) = {(bank_df['pdays']==-1).sum()}")

job has unique values: 70 || 0.63%
education has unique values: 497 || 4.45%
contact has unique values: 2346 || 21.02%
poutcome has unique values: 8326 || 74.59%

pdays=-1 (customer was not contacted in previous campaign) = 8324


In [8]:
pd.crosstab(bank_df['pdays']==-1,bank_df['poutcome']=='unknown',normalize=True) * 100

poutcome,False,True
pdays,,
False,25.407633,0.017918
True,0.000000,74.574449


- pdays = -1 acts as a sentinal value indicating that the customer was not contacted in previous campaigns. poutcome = 'unknown' makes perfect sense that as the customer was not contacted before so outcome is unknown.

## Summary Stats

In [9]:
bank_df.describe().T.style.background_gradient(cmap='Greens')

,count,mean,std,min,25%,50%,75%,max
age,11162.000000,41.231948,11.913369,18.000000,32.000000,39.000000,49.000000,95.000000
balance,11162.000000,1528.538524,3225.413326,-6847.000000,122.000000,550.000000,1708.000000,81204.000000
day,11162.000000,15.658036,8.420740,1.000000,8.000000,15.000000,22.000000,31.000000
duration,11162.000000,371.993818,347.128386,2.000000,138.000000,255.000000,496.000000,3881.000000
campaign,11162.000000,2.508421,2.722077,1.000000,1.000000,2.000000,3.000000,63.000000
pdays,11162.000000,51.330407,108.758282,-1.000000,-1.000000,-1.000000,20.750000,854.000000
previous,11162.000000,0.832557,2.292007,0.000000,0.000000,0.000000,1.000000,58.000000


- Age has mean greater than median indicating slight right skewed distribution. Its std of approx 12 years show the varability in age distribution around the mean. Age has a range of 77 years from min to max indicating a wide spread in ages. Max age of 95 years indicate that there are few customers who are relatively older than the typical customers potentialy creating a long right tail distribution.

- Balance has a mean significantly greater than median indicating strong right skewed distribution. Its std of 3225.41 USD shows significant varability is account balance distribution around the mean. Negative balance is not invalid, could be caused beacuse of over-draft or other financial obligations. Max balance indicate that at-least once customer has significantly high balance while rest of the customers have low balance potentialy creating a long right tail effect.

- Day has a mean closer to median indicating near normal distribution, but this alone is not sufficent to assume that distribution follows gaussion distribution. Day has a range of 30 days from min to max indicating the number of days in a calender month. Its std of 8.4 days show moderate varability in the day which customer was contacted from the mean.

- Duration has a mean sigificantly greater than median indicating strong right skewed distribution. Its std indicates significant varability in the duration of call around the mean. Duration has a range of 3879 secs from min to max indicating the wide spread in duration of calls also max duration suggest that at-least once call lasted relatively longer from other calls potentialy creating a long right tail distribution. Note: Duration will introduce bias and poor model performance as we only came to know about the duration of the call once the call was taken.

- Campaign has a mean greater than median indicating right skewed distribution. Its std indicates moderate varability in number of contacts made during the campaign from mean. It has a range of 62 contacts from min t0 max indicating wide spread distribution. Max campaign indicates that alteast one customer was contacted relatively high from the rest of the contacts potentialy creating a long right tail distribution.

- pdays = -1 indicates sentinal value indicating the customer was not contacted in previous campaigns. Pdays has a mean much greater than median indicating strong right skewed distribution. Its std indicate significant varability in the number of days passed since last contact. pdays has a wide spread according to its huge range from min to max. Max value indicate that atleast once customer was not contacted since 854 days from the last contact.

- previous has a mean greater than median indicating a strong right skewed distribution. Its std show high varability in number of previous contacts made around the mean. Max value of previous indicates atleast once customer was contacted repeatedly while others customer contact count sit close to zero potentialy creating a long right tail distribution.

## Univariate Analysis

In [10]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
import numpy as np

for column in bank_df.select_dtypes(include='number').columns:
    data = bank_df[column].to_numpy()

    skewness = bank_df[column].skew()
    if skewness > 0.5:
        skew = 'Right Skewed'
    elif skewness < -0.5:
        skew = 'Left Skewed'
    else:
        skew = 'Approximately Symmetric'

    q1,q3 = np.percentile(bank_df[column],[25,75])
    iqr = q3-q1
    lo,hi = q1-1.5*iqr, q3+1.5*iqr 
    n_outlier = ((bank_df[column] < lo) | (bank_df[column] > hi)).sum()

    fig = px.histogram(
        data_frame=bank_df,
        x=column,
        histnorm='probability density',
        marginal='box',
        title=f"{column.title()} --> skewness: {skewness:.2f} ({skew}) | #Outlier: {n_outlier}"
    )

    kde = gaussian_kde(data)
    xvalues = np.linspace(data.min(),data.max(),500)
    kde_counts = kde(xvalues)

    fig.add_traces(
        go.Scatter(
            x=xvalues,
            y=kde_counts,
            mode = 'lines',
            name='KDE'
        )
    )

    fig.update_layout(
        title_x = 0.5,
        xaxis_title = column.title(),
        yaxis_title = 'Probability Density',
        width = 900,
        height = 600
    )

    fig.show()

- Except age and day every other column is heavily right skewed. Balance has a skewness of 8.22 which is likely because of the extreme balances. The skewness of the columns can be reduced using different transformation techniques if the model doesn't generalize on true data.

In [11]:
for column in bank_df.select_dtypes(include='object').columns:
    if column == 'deposit':
        continue
    value_count = bank_df[column].value_counts().sort_values(ascending=False).reset_index(name='counts')

    fig = px.bar(
        data_frame=value_count,
        x=column,
        y='counts',
        color='counts',
        title=f"{column.title()} Distribution"
    )

    fig.update_layout(
        title_x = 0.5,
        xaxis_title = column.title(),
        yaxis_title = f'{column.title()} counts',
        width=900,
        height=600
    )

    fig.show()

 - Job and Month have 12 categories each perfect for onehot encoding. 

## Mutlivariate Analysis

In [12]:
for column in bank_df.select_dtypes(include='number').columns:
    fig = px.box(
        data_frame=bank_df,
        y=column,
        x='deposit',
        color='deposit',
        title=f"Number of deposit by {column.title()}",
    )

    fig.update_layout(
        title_x = 0.5,
        xaxis_title = 'Deposit',
        yaxis_title = column.title(),
        width = 900,
        height = 600
    )

    fig.show()

- Duration perfectly splits yes and no class, this is because we only came to know about the duration of the call once the call is taken. This will introduce data leakage and poor model performance. Drop this column while training machine learning model.

In [14]:
base_rate = (bank_df['deposit'] == 'yes').mean()

for column in bank_df.select_dtypes(include='object').columns:
    if column == 'deposit':
        continue
    conversion_rate = bank_df.groupby(column)['deposit'].apply(
        lambda x:(x == 'yes').mean()
    ).sort_values(ascending=False).reset_index(name='conversion_rate')

    fig = px.bar(
        data_frame=conversion_rate,
        x=column,
        y='conversion_rate',
        color='conversion_rate',
        title=f'Conversion rate by: {column.title()}'
    )

    fig.add_hline(
        y=base_rate,
        line_dash = 'dash',
        line_color = 'red',
        annotation_text=f'Base rate: {base_rate:.2%}',
        annotation_position='right top',
    )

    fig.update_layout(
        title_x = 0.5,
        xaxis_title = column.title(),
        yaxis_title = 'Conversion Rate',
        yaxis = dict(tickformat='.0%'),
        width=900,
        height=600
    )

    fig.show()

- In job (student and retired) have high conversion rate well above the base rate. poutcome = success has higher conversion rate indicating that customer who have already opted term deposit in past have higher conversion rate.

## Multicollinearity Check

In [17]:
corr_matrix = bank_df[bank_df.select_dtypes(include='number').columns].corr()
mask = np.triu(np.ones_like(corr_matrix,dtype=bool))
corr_mask = corr_matrix.mask(mask)

fig = px.imshow(
    img=corr_mask,
    zmax=1,
    zmin=-1,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='tealrose',
    title=f"Correlation Matrix",
)

fig.update_layout(
    title_x = 0.5
)

fig.show()

 - pdays and previous are only 2 coloumns showing correlation. We can drop one column if the model doesn't generalize on true data to improve the model efficency. 


## Feature Engineering

In [ ]:
fe = bank_df.copy()

In [19]:
fe['balance_log'] = np.sign(bank_df['balance']) * np.log1p(np.abs(bank_df['balance']))
fe['campaign_log'] = np.log1p(bank_df['campaign'])
fe['previous_log'] = np.log1p(bank_df['previous'])

In [20]:
fe.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,balance_log,campaign_log,previous_log
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes,7.759614,0.693147,0.0
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes,3.828641,0.693147,0.0
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes,7.147559,0.693147,0.0
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes,7.814803,0.693147,0.0
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes,5.220356,1.098612,0.0


In [21]:
## dropping unwanted columns
columns = ['balance','duration','campaign','previous']
fe.drop(columns=columns,inplace=True)

fe.head()

,age,job,marital,education,default,housing,loan,contact,day,month,pdays,poutcome,deposit,balance_log,campaign_log,previous_log
0,59,admin.,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.759614,0.693147,0.0
1,56,admin.,married,secondary,no,no,no,unknown,5,may,-1,unknown,yes,3.828641,0.693147,0.0
2,41,technician,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.147559,0.693147,0.0
3,55,services,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.814803,0.693147,0.0
4,54,admin.,married,tertiary,no,no,no,unknown,5,may,-1,unknown,yes,5.220356,1.098612,0.0


In [22]:
## saving the transformed dataframe to the local dir
fe.to_csv('data/fe_bank.csv',index=False)